# 060 — Site case-study MSA collapse fragilities

Collects the Multiple-Stripe-Analysis (MSA) collapse fragility of each case-study
structure from the analysis drive, copies it into the processed-data tree **with a
provenance manifest**, plots it against its stripe IMLs, and reports which curves are
*not well defined* — i.e. whose stripes never bracket collapse.

The processed copies written here are what notebook **061** consumes.

## Inputs

- `D:/07_wp1_casestudy_sites/site_{i}/{n}s/mdof/msa_AvgSA_03/collapse_fragility.json` —
  the MSA run output (`median`, `dispersion`, and the 2xN empirical curve `efc`).
- `data_processed/05_gcim_distributions/imls_for_selection_AvgSA_03.json` — the stripe
  IMLs, drawn on the plots as guide-lines.

## Outputs

| File | Content |
| --- | --- |
| `data_processed/09_structure_fragility_curves/wp1_casestudy_sites/site_{i}/{tag}_msa_collapsefragility_AvgSA_03.json` | the copied fragility (read by nb 061) |
| `.../{tag}_msa_collapsefragility_AvgSA_03.json.manifest.json` | its provenance sidecar |
| `results/05_site_fragility_curves/fragility_curves_site_{i}_{n}s.jpg` | the fragility plot |

## Caching

Each structure is cached against the **content hash of its source
`collapse_fragility.json`**, via `cache_utils.json_load_or_compute` — the analysed IMLs
live inside that file as `efc[0]`, so its hash already pins which stripes the curve was
fitted to. A structure whose `.manifest.json` still matches is
left untouched — the JSON is not rewritten and the figure is not redrawn (unless the JPG
has gone missing). A re-run after re-running a handful of MSA analyses therefore
reprocesses only those. Set `FORCE_RECOMPUTE = True` to rebuild everything.

Section 5 then decides what to do about the curves whose stripes failed to bracket
collapse, and writes the two remediation lists that notebooks 017 and 050 read back.

> **Run order.** The MSA results live on `D:` — run this on the machine with the drive
> attached to (re)build the processed copies. Without it the notebook prints a warning
> and falls back to the copies already in `data_processed/09_.../`, so the plots and the
> well-definedness report still work, but nothing is written.

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Setup & parameters

In [30]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

from phd_project.config import config
from phd_project.scripts.cache_utils import fingerprint, json_load_or_compute

cfg = config.load_config()

In [31]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
GM_SET = "AvgSA_03"
N_STOREYS = [5]              # structure types to process
SITES = list(range(0, 60))      # case-study site indices

# --- FOLDERS ---
ANALYSIS_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]   # D: - the MSA run output
FRAG_ROOT = cfg["proc_data"]["wp1_sites_fragility_curves"]    # the processed JSON copies
RESULTS_ROOT = cfg["results"]["site_fragility_curves"]        # the fragility JPGs
STRIPE_IML_PATH = cfg["proc_data"][f"{GM_SET}_imls_for_selection"]

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# --- WELL-DEFINEDNESS THRESHOLDS ---
# A curve is well defined only if its stripes bracket collapse: the empirical curve must
# rise ABOVE PC_UPPER_THRESHOLD and start BELOW PC_LOWER_THRESHOLD.
#
# A tail counts as reached only by a stripe strictly INSIDE (0, PC_LOWER_THRESHOLD) or
# (PC_UPPER_THRESHOLD, 1). A degenerate stripe - 0/30 or 30/30 records collapsed - bounds the
# crossing without locating it, so it satisfies nothing on its own; one sitting alongside an
# informative stripe is harmless. With 30 records the attainable non-degenerate values run
# 1/30 = 0.033 to 29/30 = 0.967, so a qualifying stripe is always reachable.
PC_UPPER_THRESHOLD = 0.8
PC_LOWER_THRESHOLD = 0.2

# --- REMEDIATION BANDS ---
# Where a *remediating* stripe must land, tighter than the pass/fail thresholds above:
# 2-5 and 25-28 collapses out of the 30 records per stripe, i.e. far enough from an
# all-collapse / no-collapse stripe that one more MSA round is definitely enough.
PC_BAND_LOW = (0.067, 0.167)
PC_BAND_HIGH = (0.833, 0.933)

# Every IML in this pipeline is stored rounded to 4 dp, while the band edges below are
# computed from a continuous lognormal. Compare the two with half an ulp of slack, or an
# IML already on the grid gets missed by a hair and is reported as needing a new
# disaggregation it does not need.
IML_TOL = 5e-5

DISAGG_SIGMA = 4    # truncation level nb 017 picked the stripe IMLs at
DISAGG_GRID_PATH = cfg["proc_data"][f"disagg_imls_{GM_SET}"]
HAZARD_CURVE_PATH = cfg["proc_data"][f"{GM_SET}_hazard_curves_{DISAGG_SIGMA}sig"]

# --- REMEDIATION OUTPUTS (written by section 5) ---
# Both are read back by nb 017 (additional IMLs -> the disaggregation grid and the per-site
# record-selection lists) and nb 050 (reserve stripes -> the MSA run list).
RESERVE_STRIPE_PATH = cfg["proc_data"][f"{GM_SET}_reserve_stripes"]
ADDITIONAL_IML_PATH = cfg["proc_data"][f"{GM_SET}_additional_imls"]

# Escape hatch: re-copy and re-plot every structure, ignoring the per-structure
# manifests. Leave False for normal incremental runs (only new/changed MSA runs rebuild).
FORCE_RECOMPUTE = True


def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"


print(f"ANALYSIS_ROOT = {ANALYSIS_ROOT}")
print(f"FRAG_ROOT     = {FRAG_ROOT}")
print(f"RESULTS_ROOT  = {RESULTS_ROOT}")

ANALYSIS_ROOT = D:\07_wp1_casestudy_sites
FRAG_ROOT     = C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites
RESULTS_ROOT  = C:\Users\clemettn\Documents\phd\results\05_site_fragility_curves


## 1. Stripe IMLs

The IMLs each structure's MSA stripes were run at, keyed by site index then structure
tag. They are drawn on the fragility plots as guide-lines and folded into the cache
fingerprint, so editing the IML file redraws the affected figures.

In [32]:
with open(STRIPE_IML_PATH, "r") as file:
    stripe_imls = json.load(file)

print(f"{len(stripe_imls)} sites in {STRIPE_IML_PATH.name}")

60 sites in imls_for_selection_AvgSA_03.json


## 2. Analysis-root availability

`ANALYSIS_ROOT` lives on the external `D:` drive. When it is not attached this is a
**warning, not an error**: the notebook switches to reading the already-processed copies
under `FRAG_ROOT` so the plots and the report below still run, and writes nothing.

In [33]:
ANALYSIS_OK = ANALYSIS_ROOT.exists()

if ANALYSIS_OK:
    print(f"Analysis root available: {ANALYSIS_ROOT}")
else:
    print(f"WARNING: analysis root {ANALYSIS_ROOT} is not accessible (drive not attached).")
    print(f"  Falling back to the processed copies in {FRAG_ROOT}.")
    print("  No fragility JSONs will be (re)written.")

Analysis root available: D:\07_wp1_casestudy_sites


## 3. Process the MSA fragility curves

For every `(site, storeys)`: resolve the source fragility on the analysis drive, pass it
through the provenance cache, run the well-definedness check, and redraw the figure only
when the cache was rebuilt (or the JPG is missing).

Statuses reported at the end:

- **computed** — source changed (or is new); JSON + manifest + JPG (re)written.
- **cached** — manifest matched; nothing rewritten.
- **offline** — read from the processed copy because `D:` is detached; nothing written.
- **skipped** — no MSA fragility available for that structure at all.

In [34]:
def fmt_sites(sites: list[int]) -> str:
    """Compact, sorted site listing for the printed summaries."""
    return ", ".join(str(s) for s in sorted(sites)) if sites else "-"


def msa_source_path(site: int, ns: int) -> Path:
    """The collapse fragility written by the MSA run, on the analysis drive."""
    return (ANALYSIS_ROOT / f"site_{site}" / f"{ns}s" / "mdof"
            / f"msa_{GM_SET}" / "collapse_fragility.json")


def processed_path(site: int, ns: int) -> Path:
    """The processed copy consumed by notebook 061."""
    tag = structure_tag(site, ns)
    return FRAG_ROOT / f"site_{site}" / f"{tag}_msa_collapsefragility_{GM_SET}.json"


def plot_path(site: int, ns: int) -> Path:
    return RESULTS_ROOT / f"fragility_curves_site_{site}_{ns}s.jpg"


def plot_fragility_curve(fc, site, ns, out_fp):
    """Fitted lognormal + empirical MSA points, with the analysed stripe IMLs marked.

    The guide-lines come from the empirical curve itself, so they mark the stripes that
    were actually run rather than the wider candidate list in imls_for_selection.
    """
    im_max = lognorm.ppf(0.99, s=fc["dispersion"], scale=fc["median"])
    imls = np.linspace(0, im_max * 1.15, 50)   # in g
    msa_fc_fit = lognorm.cdf(imls, s=fc["dispersion"], scale=fc["median"])

    fig, ax = plt.subplots(figsize=(6, 4))
    plt.close(fig)

    siml, spc = fc["efc"][0, :], fc["efc"][1, :]
    # 0% / 100% stripes are drawn in red: they bound the collapse crossing without locating
    # it, so section 4 does not let them satisfy a tail.
    informative = (spc > 0) & (spc < 1)

    for x in siml:
        ax.axvline(x, ls="--", color="k", alpha=0.5)

    ax.plot(imls, msa_fc_fit, color="b", label="MSA")
    ax.plot(siml[informative], spc[informative], ls="none", marker="o", mfc="b", mec="k",
            alpha=0.75, label="MSA ecdf")
    if (~informative).any():
        ax.plot(siml[~informative], spc[~informative], ls="none", marker="o", mfc="r",
                mec="k", alpha=0.75, label="degenerate stripe (0 / 100%)")

    ax.set_ylim(0, 1)
    ax.grid(ls="-.", color="0.8")
    ax.set_xlabel("AvgSA[0,3], [g]")
    ax.set_ylabel("Probability of Collapse, P[C]")
    ax.set_title(f"Fragility Curves - Site {site}, {ns}s")
    leg = ax.legend()
    leg.get_frame().set_edgecolor("k")

    fig.savefig(out_fp, dpi=300, bbox_inches="tight")

In [35]:
not_well_defined = {ns: {"lt_upper_threshold": [], "gt_lower_threshold": [],
                         "has_zero_stripe": [], "has_full_stripe": [],
                         "only_degenerate": []}
                    for ns in N_STOREYS}
assessed = {ns: [] for ns in N_STOREYS}
statuses = {"computed": [], "cached": [], "offline": [], "skipped": []}

for site in SITES:
    for ns in N_STOREYS:
        save_fp = processed_path(site, ns)
        jpg_fp = plot_path(site, ns)

        if ANALYSIS_OK:
            src = msa_source_path(site, ns)
            if not src.is_file():
                statuses["skipped"].append((site, ns))
                continue

            # The cache key is the content of the source fragility, and nothing else. The
            # analysed IMLs are inside it as efc[0], so the hash already pins exactly which
            # stripes this curve was fitted to - whereas the candidate list in
            # imls_for_selection also names stripes that were never run, and grows when
            # nb 017 merges remediation IMLs back in. Re-running the MSA is what
            # invalidates the processed copy, which is what the manifest should claim.
            save_fp.parent.mkdir(parents=True, exist_ok=True)
            fc_raw, status = json_load_or_compute(
                save_fp,
                fingerprint(msa_fragility=src),
                lambda src=src: json.load(open(src)),
                force=FORCE_RECOMPUTE,
                input_paths={"msa_fragility": src},
            )
        else:
            # Offline: reuse the processed copy as-is. Nothing is written and no manifest
            # is touched, so provenance stays whatever the last online run stamped.
            if not save_fp.is_file():
                statuses["skipped"].append((site, ns))
                continue
            with open(save_fp, "r") as file:
                fc_raw = json.load(file)
            status = "offline"

        statuses[status].append((site, ns))
        fc = {k: np.array(v) if isinstance(v, list) else v for k, v in fc_raw.items()}

        # --- well-definedness check: always runs, cached or not, so the report below is
        # complete on every run rather than only covering the rebuilt structures.
        assessed[ns].append(site)
        # A stripe where no record collapsed (P[C] = 0) or every record did (P[C] = 1)
        # bounds the collapse crossing without locating it, so it satisfies neither tail:
        # only a stripe strictly inside (0, 0.2] or [0.8, 1) counts.
        pc = fc["efc"][1, :]
        informative = (pc > 0) & (pc < 1)
        needs_high = not bool((informative & (pc >= PC_UPPER_THRESHOLD)).any())
        needs_low = not bool((informative & (pc <= PC_LOWER_THRESHOLD)).any())
        if needs_high:
            not_well_defined[ns]["lt_upper_threshold"].append(site)
        if needs_low:
            not_well_defined[ns]["gt_lower_threshold"].append(site)
        if (pc <= 0).any():
            not_well_defined[ns]["has_zero_stripe"].append(site)
        if (pc >= 1).any():
            not_well_defined[ns]["has_full_stripe"].append(site)
        # structures the old range-only test would have passed: their only stripe past a
        # threshold is a degenerate one
        if ((needs_high and pc.max() >= PC_UPPER_THRESHOLD)
                or (needs_low and pc.min() <= PC_LOWER_THRESHOLD)):
            not_well_defined[ns]["only_degenerate"].append(site)

        # --- plot only when the cache was rebuilt, or the figure has gone missing
        if status == "computed" or not jpg_fp.is_file():
            plot_fragility_curve(fc, site, ns, jpg_fp)

print("\n" + "   ".join(f"[{name}] {len(v)}" for name, v in statuses.items()))
if statuses["skipped"]:
    print("  skipped (no MSA fragility available):")
    for ns in N_STOREYS:
        sk = [s for s, n in statuses["skipped"] if n == ns]
        if sk:
            listing = "all sites" if len(sk) == len(SITES) else fmt_sites(sk)
            print(f"    {ns}s - {len(sk)}: {listing}")


[computed] 57   [cached] 0   [offline] 0   [skipped] 3
  skipped (no MSA fragility available):
    5s - 3: 24, 31, 38


## 4. Fragility-curve definition check

Which structures the MSA stripes failed to bracket collapse for, **reported separately
for each storey count** — the two structure types have different periods and base-shear
coefficients, so their stripe placement fails in different ways and the counts are only
meaningful when kept apart.

### Degenerate stripes do not count

A stripe where **no** record collapsed (P[C] = 0) or **every** record did (P[C] = 1) says
only that the collapse crossing is somewhere above or below it — it does not locate it, and
the binomial fit gets nothing from it beyond a bound. So a tail is reached only by a stripe
**strictly inside** `(0, PC_LOWER_THRESHOLD)` or `(PC_UPPER_THRESHOLD, 1)`:

- `needs_low` — no stripe has `0 < P[C] < 0.2`;
- `needs_high` — no stripe has `0.8 < P[C] < 1`.

A degenerate stripe sitting *alongside* an informative one is harmless; only a structure
whose sole stripes past a threshold are degenerate fails on this account, and §5 then
remediates it like any other. With 30 records the attainable non-degenerate values run
1/30 = 0.033 … 29/30 = 0.967, so a qualifying stripe always exists. The report below counts
the degenerate stripes and names the structures whose verdict they changed, and the
fragility figures draw those points in red.

In [36]:
print("Fragility-curve definition check")
print(f"  thresholds: some stripe must land in ({PC_UPPER_THRESHOLD:.2f}, 1) and some "
      f"stripe in (0, {PC_LOWER_THRESHOLD:.2f})")
print("  0% / 100% stripes are degenerate - they bound the crossing without locating it, "
      "so they satisfy neither tail")

for ns in N_STOREYS:
    n_assessed = len(assessed[ns])
    print(f"\n{ns}s structures - {n_assessed} assessed")

    if n_assessed == 0:
        print("  no MSA fragility curves found.")
        continue

    lt = not_well_defined[ns]["lt_upper_threshold"]
    gt = not_well_defined[ns]["gt_lower_threshold"]

    print(f"  no stripe in [{PC_UPPER_THRESHOLD:.2f}, 1) (upper tail not reached) - "
          f"{len(lt)} sites:")
    print(f"    {fmt_sites(lt)}")
    print(f"  no stripe in (0, {PC_LOWER_THRESHOLD:.2f}] (lower tail not reached) - "
          f"{len(gt)} sites:")
    print(f"    {fmt_sites(gt)}")

    zero = not_well_defined[ns]["has_zero_stripe"]
    full = not_well_defined[ns]["has_full_stripe"]
    only_deg = not_well_defined[ns]["only_degenerate"]
    print(f"  degenerate stripes: {len(zero)} site(s) with a 0% stripe, {len(full)} with a "
          f"100% stripe; {len(only_deg)} fail a tail only because of one:")
    print(f"    {fmt_sites(only_deg)}")

    n_bad = len(set(lt) | set(gt))
    print(f"  well defined: {n_assessed - n_bad}/{n_assessed}  "
          f"({n_bad} site(s) fail at least one threshold)")

Fragility-curve definition check
  thresholds: some stripe must land in (0.80, 1) and some stripe in (0, 0.20)
  0% / 100% stripes are degenerate - they bound the crossing without locating it, so they satisfy neither tail

5s structures - 57 assessed
  no stripe in [0.80, 1) (upper tail not reached) - 35 sites:
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 23, 25, 26, 27, 29, 30, 34, 37, 42, 44, 45, 46, 53, 56
  no stripe in (0, 0.20] (lower tail not reached) - 17 sites:
    4, 12, 17, 18, 22, 23, 27, 28, 32, 33, 35, 36, 54, 55, 57, 58, 59
  degenerate stripes: 10 site(s) with a 0% stripe, 5 with a 100% stripe; 2 fail a tail only because of one:
    54, 56
  well defined: 10/57  (47 site(s) fail at least one threshold)


## 5. Stripe-coverage remediation

Section 4 says *which* structures are badly conditioned; this section says *what to do
about each one*, and separates the cheap fix from the expensive one.

**The goal.** Every structure needs at least one **non-degenerate** stripe below
`PC_LOWER_THRESHOLD` and one above `PC_UPPER_THRESHOLD`, so the MLE fit is anchored on both
tails. A 0/30 or 30/30 stripe does not anchor anything (§4), so a tail carrying only those
is treated here exactly like a tail with no stripe past the threshold at all.

The acceptance band below already keeps a remediating stripe away from 0 and 1, and
criterion 2 already refuses to re-propose an IML the structure has analysed, so nothing else
in this section needs to change for it.

**The two remedies.**

1. **Run a reserve stripe** — nb 017 reserved one extra grid IML below and one above the
   four analysed stripes, as each structure's `reserves: [low, high]` field in
   `imls_for_selection_AvgSA_03.json` (either is `null` where the hazard ceiling stopped
   it). Ground-motion ensembles already exist for these IMLs, so this costs an MSA run and
   nothing else — no disaggregation, no record selection.
2. **Add a new IML** — a fresh disaggregation (unless the IML happens to land on the
   existing 13-point disagg grid) plus GCIM and record selection, then the MSA run.

**The acceptance band.** This has to be the last remediation round, so a remediating
stripe is only accepted if its *predicted* P[C] lands in `PC_BAND_LOW` / `PC_BAND_HIGH` —
2-5 or 25-28 collapses of 30. That is deliberately stricter than the 0.2 / 0.8 pass-fail
test: a stripe predicted at 0.02 would technically clear the lower threshold but has a
real chance of coming back with zero collapses.

P[C] at a not-yet-run IML is predicted from the **lognormal already fitted to the four
analysed stripes** (`median`, `dispersion` in each `collapse_fragility.json`).

### Outputs

| File | Content |
| --- | --- |
| `RESERVE_STRIPE_PATH` | `{"3": [[site, iml], ...], "5": [...]}` - reserve stripes to run against the existing record sets. Read back by **nb 050**, which adds exactly these to each structure's MSA run list. |
| `ADDITIONAL_IML_PATH` | `{"3": [[site, iml], ...], "5": [...]}` - one entry per new IML that needs disaggregation and record selection. Read back by **nb 017**, which merges them into each structure's `additional` field, the site `union` lists and the flat disaggregation grid. |

Re-running nb 017 after this notebook therefore closes the loop; nb 020/021 and 031-036
then pick the new IMLs up as ordinary work.

JSON object keys are strings, so reload the storey count with `int(k)`. A structure appears
twice if both of its tails need fixing, and can appear in both files if one tail is covered
by a reserve and the other is not.

In [37]:
def predict_pc(fc, iml):
    """P[C] at an arbitrary IML, from the lognormal fitted to the analysed stripes."""
    return float(lognorm.cdf(iml, s=fc["dispersion"], scale=fc["median"]))


def iml_at_pc(fc, pc):
    """Inverse of predict_pc: the IML whose fitted P[C] equals pc."""
    return float(lognorm.ppf(pc, s=fc["dispersion"], scale=fc["median"]))


def max_usable_iml(hc):
    """Largest IML with strictly positive MAFE (the truncation ceiling). From nb 017."""
    nonpos = np.where(hc[:, 1] <= 0.0)[0]
    return float(hc[nonpos[0] - 1, 0] if len(nonpos) else hc[-1, 0])


# The IMLs disaggregation has already been run at (nb 017/021): a remediating stripe placed
# on one of these needs new record selection, but no new disaggregation.
DISAGG_GRID = np.loadtxt(DISAGG_GRID_PATH)

with open(HAZARD_CURVE_PATH, "rb") as file:
    hcs = pickle.load(file)

# Per-site hazard limits. The ceiling is the one nb 017 capped its IML choice with, and
# the reason nine 3s structures have no high reserve at all. The disagg grid can always be
# extended downwards, so the floor is just where the hazard curve itself starts.
iml_ceiling = {site: max_usable_iml(hcs[site]["AvgSA"]["mean"]) for site in hcs}
iml_floor = {site: float(hcs[site]["AvgSA"]["mean"][0, 0]) for site in hcs}

print(f"disagg grid ({len(DISAGG_GRID)} IMLs): {np.round(DISAGG_GRID, 3).tolist()}")
print(f"hazard curves span {min(iml_floor.values()):.4f} g to a "
      f"{DISAGG_SIGMA}-sigma ceiling of {min(iml_ceiling.values()):.2f} - "
      f"{max(iml_ceiling.values()):.2f} g")

disagg grid (24 IMLs): [0.164, 0.211, 0.242, 0.27, 0.285, 0.31, 0.33, 0.36, 0.4, 0.45, 0.49, 0.5, 0.55, 0.605, 0.65, 0.714, 0.789, 0.8, 0.89, 0.95, 1.011, 1.15, 1.337, 1.804]
hazard curves span 0.0005 g to a 4-sigma ceiling of 0.49 - 2.09 g


In [38]:
# Re-read the processed copies rather than reusing anything from section 3, so this section
# stands on its own whether or not the cache was hit above.
diagnostics = []
for site in SITES:
    for ns in N_STOREYS:
        save_fp = processed_path(site, ns)
        if not save_fp.is_file():
            continue
        with open(save_fp, "r") as file:
            fc = json.load(file)

        efc = np.array(fc["efc"])
        informative = (efc[1, :] > 0) & (efc[1, :] < 1)
        # nb 017 writes the reserves as [low, high]; either is null where the trial grid ran
        # out or the neighbour sat above the site's hazard ceiling.
        lo_res, hi_res = stripe_imls[str(site)][structure_tag(site, ns)]["reserves"]
        diagnostics.append({
            "site": site,
            "ns": ns,
            "median": fc["median"],
            "dispersion": fc["dispersion"],
            "pc_min": efc[1, :].min(),
            "pc_max": efc[1, :].max(),
            # a 0/30 or 30/30 stripe bounds the crossing without locating it, so it
            # satisfies neither tail - see section 4
            "n_zero": int((efc[1, :] <= 0).sum()),
            "n_full": int((efc[1, :] >= 1).sum()),
            "needs_low": not bool((informative & (efc[1, :] <= PC_LOWER_THRESHOLD)).any()),
            "needs_high": not bool((informative & (efc[1, :] >= PC_UPPER_THRESHOLD)).any()),
            "analysed": efc[0, :].tolist(),
            "lo_reserve": lo_res,
            "hi_reserve": hi_res,
            "pc_lo_reserve": predict_pc(fc, lo_res) if lo_res is not None else np.nan,
            "pc_hi_reserve": predict_pc(fc, hi_res) if hi_res is not None else np.nan,
            "iml_ceiling": iml_ceiling[site],
            "iml_floor": iml_floor[site],
        })

diag = pd.DataFrame(diagnostics)
print(f"{len(diag)} structures assessed - {diag.needs_low.sum()} need a lower stripe, "
      f"{diag.needs_high.sum()} need an upper one, "
      f"{(diag.needs_low & diag.needs_high).sum()} need both")

diag

57 structures assessed - 17 need a lower stripe, 35 need an upper one, 5 need both


,site,ns,median,dispersion,pc_min,pc_max,n_zero,n_full,needs_low,needs_high,analysed,lo_reserve,hi_reserve,pc_lo_reserve,pc_hi_reserve,iml_ceiling,iml_floor
0,0,5,0.374105,0.280749,0.166667,0.600000,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.122702,0.744706,0.703768,0.0005
1,1,5,0.376624,0.170682,0.066667,0.666667,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.025590,0.851497,1.453032,0.0005
2,2,5,0.408853,0.234550,0.066667,0.433333,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.038441,0.658669,0.703768,0.0005
3,3,5,0.431429,0.248289,0.066667,0.400000,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.029537,0.567391,0.703768,0.0005
4,4,5,0.361502,0.368948,0.233333,0.600000,0,0,True,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.214465,0.723585,0.489786,0.0005
5,5,5,0.363129,0.228236,0.133333,0.633333,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.097079,0.826332,1.011236,0.0005
6,6,5,0.381127,0.186645,0.033333,0.566667,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.032382,0.813268,0.703768,0.0005
7,7,5,0.430488,0.238207,0.066667,0.533333,0,0,False,True,"[0.31, 0.36, 0.4, 0.45]",0.285,0.50,0.041691,0.735126,1.011236,0.0005
8,8,5,0.411535,0.257891,0.066667,0.400000,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.051098,0.635507,0.489786,0.0005
9,9,5,0.391396,0.227260,0.100000,0.566667,0,0,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.051151,0.730378,0.703768,0.0005


### 5.1 Criterion 1 — structures the reserve stripes can rescue

A tail is rescued by its reserve when the reserve exists **and** its predicted P[C] lands
inside the band. A structure is rescued when every tail it needs is either already covered
by an analysed stripe or rescued by a reserve; those reserve stripes go into
`RESERVE_STRIPE_PATH` and can be run against the ensembles already on disk.

Reserves that miss the band are handed to criterion 2, but the *near misses* — a reserve
that overshoots the band yet still crosses the 0.2 / 0.8 threshold — are printed
separately. Those would very probably still yield a usable stripe, and running one is far
cheaper than a fresh disaggregation, so they are worth a look before section 5.2 is acted
on.

In [39]:
def in_band(pc, band):
    return band[0] <= pc <= band[1]


reserve_to_run = {ns: [] for ns in N_STOREYS}   # (site, iml) - existing records, MSA only
unsolved = []                                   # tails that no reserve can cover
near_misses = []                                # reserve outside the band but still useful

for row in diagnostics:
    tails = (
        ("low", row["needs_low"], row["lo_reserve"], row["pc_lo_reserve"],
         PC_BAND_LOW, PC_LOWER_THRESHOLD),
        ("high", row["needs_high"], row["hi_reserve"], row["pc_hi_reserve"],
         PC_BAND_HIGH, PC_UPPER_THRESHOLD),
    )
    for tail, needs, reserve, pc_reserve, band, threshold in tails:
        if not needs:
            continue
        if reserve is not None and in_band(pc_reserve, band):
            reserve_to_run[row["ns"]].append((row["site"], reserve))
            continue
        # Not trustworthy enough to bet the last MSA round on. Flag it anyway if it would
        # still have crossed the pass/fail threshold, then hand the tail to criterion 2.
        if reserve is not None and ((tail == "low" and pc_reserve < threshold)
                                    or (tail == "high" and pc_reserve > threshold)):
            near_misses.append({**row, "tail": tail, "pc_reserve": pc_reserve})
        unsolved.append({**row, "tail": tail, "band": band})

needy = {(r["site"], r["ns"]) for r in diagnostics if r["needs_low"] or r["needs_high"]}
still_bad = {(r["site"], r["ns"]) for r in unsolved}
rescued = needy - still_bad

print("Criterion 1 - can the reserve stripes rescue the curve?")
print(f"  accepting a reserve only if its predicted P[C] is in {PC_BAND_LOW} (low) "
      f"or {PC_BAND_HIGH} (high)")

for ns in N_STOREYS:
    n_total = sum(r["ns"] == ns for r in diagnostics)
    n_needy = sum(n == ns for _, n in needy)
    n_rescued = sum(n == ns for _, n in rescued)
    print(f"\n{ns}s structures - {n_total} assessed")
    print(f"  already bracketed          : {n_total - n_needy}")
    print(f"  rescued by reserve stripes : {n_rescued}")
    print(f"  still need a new IML       : {n_needy - n_rescued}")
    print(f"  reserve stripes to run ({len(reserve_to_run[ns])}):")
    print(f"    {fmt_sites([s for s, _ in reserve_to_run[ns]])}")

print("\nNear misses - reserve outside the band but still past the threshold "
      f"({len(near_misses)}):")
for row in sorted(near_misses, key=lambda r: (r["ns"], r["site"])):
    reserve = row["lo_reserve"] if row["tail"] == "low" else row["hi_reserve"]
    print(f"  site {row['site']:>2} {row['ns']}s  {row['tail']:>4} reserve "
          f"{reserve:.3f} g -> P[C] {row['pc_reserve']:.3f}")

Criterion 1 - can the reserve stripes rescue the curve?
  accepting a reserve only if its predicted P[C] is in (0.067, 0.167) (low) or (0.833, 0.933) (high)

5s structures - 57 assessed
  already bracketed          : 10
  rescued by reserve stripes : 10
  still need a new IML       : 37
  reserve stripes to run (11):
    1, 23, 29, 32, 33, 35, 36, 44, 45, 46, 55

Near misses - reserve outside the band but still past the threshold (11):
  site  5 5s  high reserve 0.450 g -> P[C] 0.826
  site  6 5s  high reserve 0.450 g -> P[C] 0.813
  site 10 5s  high reserve 0.550 g -> P[C] 0.815
  site 14 5s  high reserve 0.450 g -> P[C] 0.809
  site 21 5s  high reserve 0.550 g -> P[C] 0.817
  site 34 5s  high reserve 0.950 g -> P[C] 0.948
  site 37 5s  high reserve 0.650 g -> P[C] 0.819
  site 42 5s  high reserve 0.800 g -> P[C] 0.988
  site 53 5s  high reserve 0.950 g -> P[C] 0.973
  site 54 5s   low reserve 0.360 g -> P[C] 0.002
  site 56 5s  high reserve 0.800 g -> P[C] 1.000


### 5.2 Criterion 2 — the minimum set of new IMLs

For every tail no reserve can cover, the fitted lognormal gives the **interval of IMLs**
that would land inside the band, `[iml_at_pc(band_lo), iml_at_pc(band_hi)]`. Each interval
is then resolved in order of cost:

**Exhausted tails are removed first.** A capped tail (case 1 below) whose pinned IML is
already one of the structure's analysed stripes has nothing left to run: the pin is the only
IML that tail can ever be given, so once its stripe has come back it is the *hazard curve*,
not the stripe placement, that keeps the band out of reach. Those tails are reported on
their own and enter none of the three cases below. Without this the pin is thrown out by the
already-analysed exclusion in case 2 and re-emerges from case 3 as a fresh disaggregation -
which is how 0.4898 g and 1.0112 g were re-proposed after they had already been run.

1. **Capped** — the whole interval sits outside the site's hazard curve, in practice
   above its truncation ceiling. The band cannot be reached, so the stripe is pinned to
   the ceiling itself, the highest IML the site can be disaggregated at, and joins the
   pool as a fixed point. It is reported separately because the resulting stripe will
   still fall short of the band. Falling below the bottom of the current disagg grid is
   *not* a cap — that grid can be extended downwards.
2. **Snap to the disagg grid** — if a grid IML falls inside the interval, use the one
   nearest the band midpoint. Disaggregation already exists there, so only record
   selection is needed. IMLs the structure has already run, or already holds as a reserve,
   are excluded: the fitted curve can sit inside the band at an IML whose actual stripe
   came back below the threshold, and re-running that stripe would change nothing.
3. **New IMLs, pooled and minimised** — the remaining intervals from *all* sites and both
   storey counts are pooled and solved as a minimum interval-stabbing problem: sort by
   upper bound, place a point at the upper bound of the first interval not yet hit, drop
   every interval that point covers. The greedy right-endpoint sweep is provably optimal,
   so the result is the fewest distinct IMLs that give every remaining structure a stripe
   inside its band.

In [40]:
def stab_intervals(intervals):
    """Fewest points hitting every interval - greedy sweep on the right endpoints.

    Sorting by upper bound and committing a point at the first uncovered upper bound is
    the classic optimal solution, so len(result) is the true minimum.
    """
    points = []
    for lo, hi in sorted(intervals, key=lambda iv: iv[1]):
        if not points or points[-1] < lo:
            points.append(hi)
    return points


def has_records(row, iml):
    """True when a record ensemble already exists for this structure at `iml`.

    nb 017 writes every IML records were selected at into the structure's `stripes` and
    `additional` lists, so an IML found there needs only the MSA run - no disaggregation
    and no record selection.
    """
    entry = stripe_imls[str(row["site"])][structure_tag(row["site"], row["ns"])]
    have = list(entry["stripes"]) + list(entry["additional"])
    return bool(np.any(np.isclose(have, iml, atol=IML_TOL)))


# 1. reachability - the hazard curve is the only hard limit. A band above the truncation
# ceiling cannot be reached at all, so that tail is *capped*: it collapses to the single
# point at the ceiling, the highest IML the site can be disaggregated at, and is carried
# through as a degenerate interval so sites sharing a ceiling share one new IML. A band
# below the bottom of the disagg grid needs no cap - the grid can be extended downwards.
#
# A capped tail whose pin has already been analysed is *exhausted*: the pin is the only IML
# it can ever be given, and its stripe is in. Nothing further can be run for it, so it is
# dropped here rather than being handed to the grid snap - which would reject it as already
# analysed and let step 3 re-propose it as a disaggregation that has already been done.
capped, exhausted, to_place = [], [], []
for row in unsolved:
    iml_lo, iml_hi = iml_at_pc(row, row["band"][0]), iml_at_pc(row, row["band"][1])
    row = {**row, "iml_lo": iml_lo, "iml_hi": iml_hi}
    if iml_lo > row["iml_ceiling"] or iml_hi < row["iml_floor"]:
        # round to the stored precision, so the pinned IML is the one that ends up in the
        # JSON rather than a value a hair away from it
        limit = round(row["iml_ceiling"] if iml_lo > row["iml_ceiling"]
                      else row["iml_floor"], 4)
        if np.any(np.isclose(row["analysed"], limit, atol=IML_TOL)):
            exhausted.append({**row, "iml": limit})
            continue
        capped.append(row)
        to_place.append({**row, "iml_lo": limit, "iml_hi": limit})
        continue
    to_place.append({**row,
                     "iml_lo": max(iml_lo, row["iml_floor"]),
                     "iml_hi": min(iml_hi, row["iml_ceiling"])})

# 2. grid snap - cheapest of the remaining options, no new disaggregation. A grid IML the
# structure has already got is no remedy at all: its stripe has been run and came back
# below the threshold, and a reserve of its own was already judged in criterion 1. The
# fitted curve can sit above the band there while the stripe itself did not, so without
# this exclusion the notebook would propose re-running a stripe and changing nothing.
grid_hits, remaining = [], []
for row in to_place:
    already = set(row["analysed"]) | {x for x in (row["lo_reserve"], row["hi_reserve"])
                                      if x is not None}
    in_range = DISAGG_GRID[(DISAGG_GRID >= row["iml_lo"] - IML_TOL)
                           & (DISAGG_GRID <= row["iml_hi"] + IML_TOL)]
    cand = np.array([x for x in in_range
                     if not np.any(np.isclose(list(already), x))], dtype=float)
    if len(cand):
        target = iml_at_pc(row, sum(row["band"]) / 2)
        grid_hits.append({**row, "source": "grid",
                          "iml": float(cand[np.argmin(np.abs(cand - target))])})
    else:
        remaining.append(row)

# 3. everything else pooled into one stabbing problem across sites and storey counts
stab_points = stab_intervals([(r["iml_lo"], r["iml_hi"]) for r in remaining])
new_hits = [{**row, "source": "new",
             "iml": next(p for p in stab_points
                         if row["iml_lo"] - IML_TOL <= p <= row["iml_hi"] + IML_TOL)}
            for row in remaining]

new_imls = {ns: [] for ns in N_STOREYS}
for row in grid_hits + new_hits:
    new_imls[row["ns"]].append((row["site"], row["iml"]))

# Of the grid hits, those whose IML the structure already holds a record set for cost only
# the MSA run - nb 050 picks them up on its own, since stripes_to_run() unions `stripes`
# with `additional`.
msa_only = [r for r in grid_hits if has_records(r, r["iml"])]
selection_only = [r for r in grid_hits if not has_records(r, r["iml"])]


def fmt_covered(rows):
    return ", ".join(f"{r['site']}/{r['ns']}s({r['tail']})"
                     for r in sorted(rows, key=lambda r: (r["ns"], r["site"])))


print("Criterion 2 - stripes that need an IML the reserves cannot provide")
print(f"  {len(unsolved)} unsolved tails: {len(exhausted)} exhausted at the site hazard "
      f"ceiling, {len(grid_hits)} already disaggregated ({len(msa_only)} needing only the "
      f"MSA run, {len(selection_only)} needing record selection), {len(remaining)} needing "
      f"a new IML, of which {len(capped)} are pinned to the hazard ceiling")

print(f"\nMINIMUM NUMBER OF IMLs TO DISAGGREGATE: {len(stab_points)}")
for point in stab_points:
    covered = [r for r in new_hits if r["iml"] == point]
    print(f"  {point:.4f} g - {len(covered)} structure(s): {fmt_covered(covered)}")

print(f"\nAlready disaggregated AND selected for - MSA run only ({len(msa_only)}):")
for iml in sorted({r["iml"] for r in msa_only}):
    covered = [r for r in msa_only if r["iml"] == iml]
    print(f"  {iml:.4f} g - {len(covered)} structure(s): {fmt_covered(covered)}")

print(f"\nAlready on the disagg grid, record selection only ({len(selection_only)}):")
for iml in sorted({r["iml"] for r in selection_only}):
    covered = [r for r in selection_only if r["iml"] == iml]
    print(f"  {iml:.4f} g - {len(covered)} structure(s): {fmt_covered(covered)}")

print(f"\nCapped at the hazard ceiling - the band itself is out of reach, so the stripe "
      f"goes as high as the site allows and will still fall short ({len(capped)}):")
for row in sorted(capped, key=lambda r: (r["ns"], r["site"])):
    print(f"  site {row['site']:>2} {row['ns']}s {row['tail']:>4} tail needs "
          f"{row['iml_lo']:.3f}-{row['iml_hi']:.3f} g -> pinned to "
          f"{row['iml_ceiling']:.4f} g, P[C] {predict_pc(row, row['iml_ceiling']):.3f}")

print(f"\nExhausted - the pinned stripe has already been run and the band is still out of "
      f"reach, so there is nothing further to do ({len(exhausted)}):")
for row in sorted(exhausted, key=lambda r: (r["ns"], r["site"])):
    print(f"  site {row['site']:>2} {row['ns']}s {row['tail']:>4} tail: {row['iml']:.4f} g "
          f"analysed, P[C] {predict_pc(row, row['iml']):.3f}, band "
          f"{row['band'][0]:.3f}-{row['band'][1]:.3f}")


Criterion 2 - stripes that need an IML the reserves cannot provide
  41 unsolved tails: 0 exhausted at the site hazard ceiling, 40 already disaggregated (39 needing only the MSA run, 1 needing record selection), 1 needing a new IML, of which 2 are pinned to the hazard ceiling

MINIMUM NUMBER OF IMLs TO DISAGGREGATE: 1
  0.5971 g - 1 structure(s): 56/5s(high)

Already disaggregated AND selected for - MSA run only (39):
  0.1645 g - 3 structure(s): 12/5s(low), 23/5s(low), 28/5s(low)
  0.2109 g - 4 structure(s): 17/5s(low), 18/5s(low), 22/5s(low), 27/5s(low)
  0.2423 g - 1 structure(s): 4/5s(low)
  0.2700 g - 1 structure(s): 58/5s(low)
  0.3300 g - 1 structure(s): 57/5s(low)
  0.3600 g - 1 structure(s): 59/5s(low)
  0.4898 g - 7 structure(s): 4/5s(high), 5/5s(high), 6/5s(high), 8/5s(high), 14/5s(high), 19/5s(high), 27/5s(high)
  0.5000 g - 4 structure(s): 0/5s(high), 9/5s(high), 20/5s(high), 26/5s(high)
  0.5500 g - 5 structure(s): 2/5s(high), 7/5s(high), 13/5s(high), 15/5s(high), 25/5s(h

### 5.3 Write the remediation lists

In [41]:
def dump_stripe_list(stripes, out_fp):
    """{ns: [(site, iml), ...]} merged into out_fp, IMLs rounded to the 4 dp the IML file uses.

    The file is *cumulative*. nb 017 rebuilds the disaggregation grid and the per-site
    record-selection lists from it, and nb 050 its MSA run lists, so an entry an earlier run
    wrote has to survive a run that covers fewer storey counts (N_STOREYS) or whose tail has
    since been closed by the analysis it asked for - overwriting would quietly delete the
    IMLs those notebooks have already acted on. New pairs are unioned into what is there.

    Returns the merged payload and, per storey count, how many pairs were new.
    """
    payload = {}
    if out_fp.is_file():
        with open(out_fp, "r") as file:
            payload = json.load(file)

    added = {}
    for ns, pairs in stripes.items():
        kept = {(int(site), round(float(iml), 4)) for site, iml in payload.get(str(ns), [])}
        new = {(int(site), round(float(iml), 4)) for site, iml in pairs} - kept
        added[str(ns)] = len(new)
        payload[str(ns)] = [[site, iml] for site, iml in sorted(kept | new)]

    with open(out_fp, "w") as file:
        json.dump(payload, file, indent=2)
    return payload, added


reserve_payload, reserve_added = dump_stripe_list(reserve_to_run, RESERVE_STRIPE_PATH)
new_payload, new_added = dump_stripe_list(new_imls, ADDITIONAL_IML_PATH)

# Every reserve stripe must already have a record ensemble, or "no new selection" is a lie.
for ns, pairs in reserve_to_run.items():
    for site, iml in pairs:
        assert np.any(np.isclose(stripe_imls[str(site)]["union"], iml)), \
            f"site {site} {ns}s: no record set selected at {iml} g"

for name, path, payload, added in (
        ("reserve stripes", RESERVE_STRIPE_PATH, reserve_payload, reserve_added),
        ("additional IMLs", ADDITIONAL_IML_PATH, new_payload, new_added)):
    counts = ", ".join(f"{ns}s {len(v)} ({added.get(ns, 0)} new)"
                       for ns, v in sorted(payload.items()))
    print(f"{name}: {counts}  ->  {path}")


reserve stripes: 3s 26 (0 new), 5s 14 (0 new)  ->  C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions\reserve_stripes_to_run_AvgSA_03.json
additional IMLs: 3s 35 (0 new), 5s 47 (2 new)  ->  C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions\additional_imls_for_disagg_AvgSA_03.json


In [42]:
with open(Path(r"D:\07_wp1_casestudy_sites\site_0\3s\mdof\AvgSA_03_stripes\site_0__stripe_iml_00pt310__gm_selection.pickle"), "rb") as file:
    stripe = pickle.load(file)
stripe["recs"]

metadata                                                       \
          index database               event_id           event_time   
38288     20180      ESM           IT-2012-0061  2012-10-25 23:05:24   
29413     10416      ESM  EMSC-20161026_0000077  2016-10-26 17:10:36   
7520       8783      ESM  EMSC-20160824_0000013  2016-08-24 02:33:29   
30369     11389      ESM  EMSC-20161030_0000029  2016-10-30 06:40:18   
32338     14000      ESM           GR-1997-0019  1997-11-18 13:07:38   
12019     14012      ESM           GR-1999-0001  1999-09-07 11:56:49   
39828     21812      ESM           ME-1979-0003  1979-04-15 06:19:41   
19919     22261      ESM           TK-2003-0003  2003-01-27 05:26:23   
33799     15598      ESM           IT-2009-0009  2009-04-06 01:32:40   
32874     14651      ESM           IT-1997-0006  1997-09-26 09:40:24   
29629     10634      ESM  EMSC-20161026_0000095  2016-10-26 19:18:06   
12649     14760      ESM           IT-1997-0137  1997-10-14 15:23:09   
10003     11351      ESM  EMSC-20161030_0000029  2016-10-30 06:40:18   
33817     15616      ESM           IT-2009-0009  2009-04-06 01:32:40   
13763     15897      ESM           IT-2009-0121  2009-04-09 00:52:59   
7231       8490      ESM  EMSC-20160824_0000006  2016-08-24 01:36:32   
19938     22280      ESM           TK-2003-0038  2003-05-01 00:27:04   
13751     15885      ESM           IT-2009-0121  2009-04-09 00:52:59   
40079     22080      ESM           TK-1999-0389  1999-11-11 14:41:23   
39843     21827      ESM           ME-1979-0012  1979-05-24 17:23:17   
34080     15883      ESM           IT-2009-0121  2009-04-09 00:52:59   
13475     15605      ESM           IT-2009-0009  2009-04-06 01:32:40   
10028     11378      ESM  EMSC-20161030_0000029  2016-10-30 06:40:18   
30384     11406      ESM  EMSC-20161030_0000029  2016-10-30 06:40:18   
32349     14011      ESM           GR-1999-0001  1999-09-07 11:56:49   
32621     14359      ESM           IT-1976-0030  1976-09-15 09:21:18   
78274   7004715   NGASub                7000044  YYYY-MM-DD HH:MM:00   
78268   7004709   NGASub                7000044  YYYY-MM-DD HH:MM:00   
76052   7001981   NGASub                7000017  YYYY-MM-DD HH:MM:00   
125377  7007993   NGASub                7000057  YYYY-MM-DD HH:MM:00   

                                                                           \
       station_code location_code                   trt   mag         rjb   
38288           LTR             0       Shallow Default  5.30   23.700000   
29413           CSC             0       Shallow Default  5.40   20.200000   
7520           RM33             0       Shallow Default  5.30   31.900000   
30369          PZI1             0       Shallow Default  6.50   31.350000   
32338          KYP1             0       Shallow Default  6.60   98.500000   
12019          ATH4             0       Shallow Default  5.90   14.040000   
39828           ULA             0       Shallow Default  6.90    5.560000   
19919          2301             0       Shallow Default  6.06  106.200000   
33799           CLN             0       Shallow Default  6.33   16.380000   
32874           MTL             0       Shallow Default  5.97   18.970000   
29629           MNF             0       Shallow Default  5.90    6.840000   
12649           CSC             0       Shallow Default  5.62   17.010000   
10003           FLT             0       Shallow Default  6.50   90.490000   
33817           ORC             0       Shallow Default  6.33   33.680000   
13763           FMG             0       Shallow Default  5.43   31.200000   
7231           RM33             0       Shallow Default  6.00   13.010000   
19938          1201             0       Shallow Default  6.36    9.070000   
13751           AQM             0       Shallow Default  5.43   12.300000   
40079          5401             0       Shallow Default  5.67   10.190000   
39843           BUD             0       Shallow Default  5.90    9.390000   
34080   

After running this notebook 050 should be rerun again to setup the supplementary analyses